# Personality Warp (PoC) — Google Colab 検証ノートブック（イラスト調モデル版）

同じプロンプト・同じ画像シード（`KSampler` seed）のまま、`personality_id` によってモデル中間層のポテンシャル地形をわずかに歪めたとき、出力イラストにどのような「個性」が現れるかを比較検証します。

In [ ]:
# 1. GPUの確認
!nvidia-smi

In [ ]:
# 2. ComfyUI と comfyui-personality-warp のセットアップ
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# --- プラグインの配置 ---
# (A) GitHub に push 済みの場合は以下を使用:
# !git clone https://github.com/<YOUR_USER>/comfyui-personality-warp.git custom_nodes/comfyui-personality-warp

# (B) Google Drive にフォルダを置いた場合は以下を使用:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/comfyui-personality-warp custom_nodes/comfyui-personality-warp

In [ ]:
# 3. 検証用のイラスト調モデル (Counterfeit-V3.0 fp16 / SD1.5系) をダウンロード
!mkdir -p /content/ComfyUI/models/checkpoints
!wget -q --show-progress -O /content/ComfyUI/models/checkpoints/Counterfeit-V3.0_fix_fp16.safetensors \
  https://huggingface.co/gsdf/Counterfeit-V3.0/resolve/main/Counterfeit-V3.0_fix_fp16.safetensors

In [ ]:
# 4. 同一シード・同一プロンプトでの A/B 比較実験 (Baseline vs 個性ID 3種)
import sys
import importlib.util
from pathlib import Path

sys.path.insert(0, '/content/ComfyUI')

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import nodes

pkg_root = Path('/content/ComfyUI/custom_nodes/comfyui-personality-warp')
spec = importlib.util.spec_from_file_location(
    'comfyui_personality_warp', pkg_root / '__init__.py', submodule_search_locations=[str(pkg_root)]
)
pkg = importlib.util.module_from_spec(spec)
sys.modules['comfyui_personality_warp'] = pkg
spec.loader.exec_module(pkg)
warp_node = pkg.NODE_CLASS_MAPPINGS['PersonalityWarp']()

# イラスト調モデルの読み込み
ckpt_loader = nodes.CheckpointLoaderSimple()
base_model, clip, vae = ckpt_loader.load_checkpoint('Counterfeit-V3.0_fix_fp16.safetensors')

# ==========================================
# ▼ ここを書き換えて実験できます
# ==========================================
prompt = '1girl, solo, anime illustration, digital art, clean lineart, soft lighting, upper body, detailed eyes, simple background'
negative = 'lowres, bad anatomy, bad hands, blurry, photo, realistic, 3d, watermark, text'

# 画像の乱数シード（全員共通にすることで「個性IDによる違い」だけを比較できる）
FIXED_SEED = 42

# 歪みの強さ（各層の標準偏差に対する倍率。0.0で歪みなし、大きいほど変化が大胆になる）
STRENGTH = 0.2

# 歪みの作り方 (使いたい1つだけコメントを外す)
MODE = 'low_rank'         # 低ランク摂動（LoRA風の構造的な変化が出やすい）
# MODE = 'gaussian'       # 通常のガウスノイズ（全体が均一に揺らぐ）
# MODE = 'rademacher'     # ±1の固定幅ノイズ

# どの範囲の層をいじるか (使いたい1つだけコメントを外す)
SCOPE = 'custom_range'    # 下の LAYER_START 〜 LAYER_END の範囲を細かく指定
# SCOPE = 'middle_only'   # UNetの中央(middle_block)だけ → 構図・柄が変わりやすい
# SCOPE = 'mid_band'      # 中央＋その前後 → 構図と画風の両方が変わる

# 層の深さ指定（SCOPE = 'custom_range' のとき有効 / 0.0=入力側 〜 0.5=中央 〜 1.0=出力側）
#   0.40〜0.55: 構図・キャラの造形に影響
#   0.45〜0.75: 画風・線・デフォルメに影響
#   0.65〜0.95: 塗り・色味・質感に影響
LAYER_START = 0.45
LAYER_END = 0.75

# いじるパラメータの種類 (使いたい1つだけコメントを外す)
TARGET_PART = 'attention_only'     # 画風・線・デフォルメ寄り
# TARGET_PART = 'mlp_only'         # 塗り・色味・質感寄り
# TARGET_PART = 'attention_and_mlp'# 両方
# TARGET_PART = 'all_weights'      # 対象層の重みすべて
# ==========================================

clip_enc = nodes.CLIPTextEncode()
pos_cond = clip_enc.encode(clip, prompt)[0]
neg_cond = clip_enc.encode(clip, negative)[0]

empty_latent = nodes.EmptyLatentImage().generate(512, 512, 1)[0]
ksampler = nodes.KSampler()
vae_dec = nodes.VAEDecode()

configs = [
    ('Baseline (0.0)', 'baseline', 0.0),
    ('ID: alice', 'alice', STRENGTH),
    ('ID: bob', 'bob', STRENGTH),
    ('ID: charlie', 'charlie', STRENGTH),
]

results = []
with torch.inference_mode():
    for label, pid, s in configs:
        warped_model, info = warp_node.apply_warp(
            model=base_model,
            personality_id=pid,
            strength=s,
            mode=MODE,
            layer_scope=SCOPE,
            target_part=TARGET_PART,
            layer_start=LAYER_START,
            layer_end=LAYER_END,
        )
        print(info)
        latent = ksampler.sample(
            model=warped_model,
            seed=FIXED_SEED,
            steps=20,
            cfg=7.5,
            sampler_name='euler',
            scheduler='normal',
            positive=pos_cond,
            negative=neg_cond,
            latent_image=empty_latent,
        )[0]
        img_tensor = vae_dec.decode(vae, latent)[0]
        img_np = (img_tensor[0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
        results.append((label, Image.fromarray(img_np)))

fig, axes = plt.subplots(1, len(results), figsize=(20, 5))
for ax, (label, img) in zip(axes, results):
    ax.imshow(img)
    ax.set_title(label, fontsize=14)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 5. (オプション) ComfyUI Web UI を起動してノードを直接触りたい場合
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

import subprocess, threading, time
def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = __import__('socket').socket()
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            sock.close()
            break
        sock.close()
    p = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if 'trycloudflare.com' in l:
            print('ComfyUI URL:', l[l.find('http'):].strip())

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
!python /content/ComfyUI/main.py --dont-print-server